In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "AreaAverages"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
spinup_hours = "0"

RunType = ("TRACER","WET","NSSL",spinup_hours)
# RunType = ("TRACER","WET","TEMPO",spinup_hours)

# RunType = ("TRACER","DRY","NSSL",spinup_hours)
# RunType = ("TRACER","DRY","TEMPO",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzc)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzf)

    else:  # 2D variable case
        shape = (ModelData.Ntime, 1)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output


def GetMean(variableSubset):
    variableMean = variableSubset.mean(dim=("latitude","longitude"), skipna=True).data
    return variableMean

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def MeanDBZ(variableSubset):
    # Convert from dBZ → linear Z (mm^6 m^-3)
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take mean in linear space
    variableMean = GetMean(variableSubset_power)

    # Convert mean Z → back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
def RunCalculations(varNames):
    outputDictionary={}
    
    num_times = ModelData.Ntime
    for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
            #Subsetting Data

            variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)

            if varName in ['refl10cm','refl10cm_1km']:
                variableSubset = variableSubset.where(variableSubset > 0)

            #Applying RadarDataMask
            variableSubset = variableSubset.where(RadarDataMask == True)
            
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset, fill_nan=False)
                outputDictionary[varName] = output

            #Taking Mean
            if varName in ['refl10cm','refl10cm_1km']:
                variableMean = MeanDBZ(variableSubset)  
            else:
                variableMean = GetMean(variableSubset)
                
            outputDictionary[varName][t] = variableMean

    return outputDictionary

# Notes:
# (1) may need to subset land/water later

In [ ]:
def RunAreaAverages(ModelData,varNames,name):
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName = f"outputDictionary_{name}.h5")
    
    #loading back in 
    try:
        outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
        return outputDictionary
    except Exception as e:
        print(f"Error: {e}")
        
        print("Running Calculation")
        outputDictionary = RunCalculations(varNames) #takes about 10 minutes
        #saving output
        
        DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)
        return outputDictionary

In [ ]:
###############
#Loading in MRMS RadarTimeseries
###############

def LoadRadarTimeseries(ModelData):
    """
    Build the time-series filename using ModelData and load the .pkl file.
    Creates output directory if needed.
    """

    # Build file name
    fileName = (
        f"RadarTimeseries_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )

    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType='DataAnalysis/Observation_Data', dataType='RadarComparison'),
        "RadarTimeseries"
    )
    os.makedirs(outputDir, exist_ok=True)

    # Full path to the .pkl file
    fullFilePath = os.path.join(outputDir, fileName)

    # Try to load existing file
    if os.path.exists(fullFilePath):
        print(f"Loading existing file: {fullFilePath}")
        with open(fullFilePath, "rb") as f:
            return fullFilePath, pickle.load(f)

    # No file found
    return fullFilePath, None

def Add_MRMS_RadarTimeSeries_Plot(ax, loc='lower right'):
    ax.plot([datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings], MRMS_RadarTimeseries, color='black',label='MRMS')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, frameon=True, fontsize=9, loc=loc)

fileName, list_array = LoadRadarTimeseries(ModelData)
if list_array is not None:
    MRMS_RadarTimeseries = list_array[:,2]

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
### Composite Types:
## Max Reflectivity .max(dim)
## Percentile Composite .quantile(0.5/0.9, dim)
## Threshold Composite .where(a>b).mean(dim)
## (Storm-Centered Composite)